Change the file size as you process

In [2]:
import os
import re
import pandas as pd
from bs4 import BeautifulSoup
import sqlite3  

# Base directory where SEC filings are stored
base_dir = "C:/Users/sudet/Desktop/to-be-processed" # Update this with the correct path
output_file = "sec_filings.csv"
DB_FILE = "sec_filings_1000_1500.db"

def extract_metadata(text, ticker):
    """Extract metadata such as company name, CIK, filing type, date, and additional fields from a 10-Q report."""
    metadata = {}
    
    # Extract company information
    company_name_match = re.search(r"COMPANY CONFORMED NAME:\s+(.+)", text)
    cik_match = re.search(r"CENTRAL INDEX KEY:\s+(\d+)", text)
    filing_type_match = re.search(r"FORM TYPE:\s+(\S+)", text)
    filed_date_match = re.search(r"FILED AS OF DATE:\s+(\d+)", text)
    period_of_report_match = re.search(r"PERIOD OF REPORT:\s+(\d+)", text)
    sec_file_match = re.search(r"SEC FILE NUMBER:\s+([\d-]+)", text)
    ein_match = re.search(r"IRS NUMBER:\s+(\d+)", text)
    sic_match = re.search(r"STANDARD INDUSTRIAL CLASSIFICATION:\s+(.+)", text)
    public_float_match = re.search(r"PUBLIC FLOAT:\s+\$([\d,]+)", text)
    outstanding_shares_match = re.search(r"OUTSTANDING SHARES:\s+([\d,]+)", text)
    
    # Store extracted metadata
    metadata["ticker"] = ticker  # Include the ticker from directory name
    metadata["company_name"] = company_name_match.group(1).strip() if company_name_match else "Unknown"
    metadata["cik"] = cik_match.group(1).strip() if cik_match else "Unknown"
    metadata["filing_type"] = filing_type_match.group(1).strip() if filing_type_match else "Unknown"
    metadata["date"] = filed_date_match.group(1).strip() if filed_date_match else "Unknown"
    metadata["period_of_report"] = period_of_report_match.group(1).strip() if period_of_report_match else "Unknown"
    metadata["sec_file_number"] = sec_file_match.group(1).strip() if sec_file_match else "Unknown"
    metadata["ein"] = ein_match.group(1).strip() if ein_match else "Unknown"
    metadata["industry_classification"] = sic_match.group(1).strip() if sic_match else "Unknown"
    metadata["public_float"] = public_float_match.group(1).strip() if public_float_match else "Unknown"
    metadata["outstanding_shares"] = outstanding_shares_match.group(1).strip() if outstanding_shares_match else "Unknown"

    return metadata

def clean_filing_text(text):
    soup = BeautifulSoup(text, "lxml")  
    clean_text = soup.get_text(separator=" ")  
    clean_text = re.sub(r"\s+", " ", clean_text).strip()  
    return clean_text

def create_database():
    """Initialize the SQLite database and create the filings table if it doesn't exist."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS filings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            ticker TEXT,
            company_name TEXT,
            cik TEXT,
            filing_type TEXT,
            date TEXT,
            period_of_report TEXT,
            sec_file_number TEXT,
            ein TEXT,
            industry_classification TEXT,
            public_float TEXT,
            outstanding_shares TEXT,
            filing_text TEXT,
            UNIQUE(ticker, date, filing_type)  -- Prevents duplicates
        )
    """)
    
    conn.commit()
    conn.close()

def insert_filing(metadata, filing_text):
    """Insert new filing record into the database, avoiding duplicates."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    
    try:
        cursor.execute("""
            INSERT INTO filings (ticker, company_name, cik, filing_type, date, period_of_report, 
                                sec_file_number, ein, industry_classification, public_float, 
                                outstanding_shares, filing_text)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            metadata["ticker"], metadata["company_name"], metadata["cik"], metadata["filing_type"], 
            metadata["date"], metadata["period_of_report"], metadata["sec_file_number"], 
            metadata["ein"], metadata["industry_classification"], metadata["public_float"], 
            metadata["outstanding_shares"], filing_text
        ))

        conn.commit()
        print(f"Added: {metadata['company_name']} ({metadata['filing_type']} - {metadata['date']})")
    
    except sqlite3.IntegrityError:
        print(f"Skipping duplicate: {metadata['company_name']} ({metadata['filing_type']} - {metadata['date']})")
    
    conn.close()

def process_filings(base_dir):
    """Process SEC filings and store in a database."""
    create_database()  
    
    max_size = 30_000_000
    
    for ticker in os.listdir(base_dir):
        ticker_path = os.path.join(base_dir, ticker)
        if not os.path.isdir(ticker_path):
            continue
        
        for filing_type in ["10-Q"]:
            filing_path = os.path.join(ticker_path, filing_type)
            if not os.path.exists(filing_path):
                continue
            
            for filing_folder in os.listdir(filing_path):
                submission_path = os.path.join(filing_path, filing_folder, "full-submission.txt")
                if not os.path.exists(submission_path):
                    print(f"      No submission file found.")
                    continue
                
                file_size = os.path.getsize(submission_path)
                if file_size > max_size:
                    print(f"⚠️ Skipping large file: {submission_path} ({file_size} bytes)")
                    continue
                
                with open(submission_path, "r", encoding="utf-8", errors="ignore") as f:
                    content = f.read()
                
                metadata = extract_metadata(content, ticker)
                filing_text = clean_filing_text(content)
                insert_filing(metadata, filing_text)  

# Run processing
process_filings(base_dir)

Added: Alcoa Corp (10-Q - 20230727)
Added: Alcoa Corp (10-Q - 20231026)
Added: Alcoa Corp (10-Q - 20190731)
Added: Alcoa Corp (10-Q - 20191031)
Added: Alcoa Corp (10-Q - 20201030)
Added: Alcoa Corp (10-Q - 20210729)
Added: Alcoa Corp (10-Q - 20211028)
Added: Alcoa Corp (10-Q - 20220725)
Added: Alcoa Corp (10-Q - 20221027)


KeyboardInterrupt: 